In [1]:
import pandas as pd
import sqlite3

# 1. Read the cleaned sheet from the uploaded Excel file
file_name = "Dataset for Data Analytics week 3.xlsx"
df = pd.read_excel(file_name, sheet_name="Cleaned Dataset")

# 2. Establish a connection to a local SQL database
conn = sqlite3.connect("ecommerce.db")

# 3. Save the dataframe data directly into a SQL table named 'orders'
df.to_sql("orders", conn, if_exists="replace", index=False)

print("✅ Success: The dataset is now a functional SQL table named 'orders'!")

✅ Success: The dataset is now a functional SQL table named 'orders'!


In [2]:
# Function to cleanly print our SQL queries as structured dataframes
def run_query(query):
    return pd.read_sql_query(query, conn)

# Query 1: Basic Selection (Preview first 5 rows)
print("--- 1. PREVIEWING DATASETRows (SELECT) ---")
display(run_query("SELECT * FROM orders LIMIT 5;"))

# Query 2: Filtering and Sorting (High value orders ordered by price)
print("\n--- 2. FILTERING & SORTING (WHERE & ORDER BY) ---")
display(run_query("SELECT OrderID, Product, Quantity, TotalPrice FROM orders WHERE TotalPrice > 2000 ORDER BY TotalPrice DESC LIMIT 5;"))

# Query 3: Basic Aggregations (Total Revenue, Avg Price, Total Orders)
print("\n--- 3. DATA AGGREGATIONS (SUM, AVG, COUNT) ---")
display(run_query("SELECT COUNT(OrderID) as Total_Orders, SUM(TotalPrice) as Total_Revenue, AVG(UnitPrice) as Avg_Unit_Price FROM orders;"))

# Query 4: Grouping Data (Sales performance broken down by product category)
print("\n--- 4. GROUPING DATA (GROUP BY) ---")
display(run_query("SELECT Product, SUM(Quantity) as Units_Sold, SUM(TotalPrice) as Revenue FROM orders GROUP BY Product ORDER BY Revenue DESC;"))

--- 1. PREVIEWING DATASETRows (SELECT) ---


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice,VerifiedPrice
0,ORD200152,45659,C77736,Chair,2,399.04,659 Main St,Cash,Cancelled,TRK90329250,3,FREESHIP,Referral,798.08,798.08
1,ORD201141,45704,C11998,Chair,3,498.66,382 Main St,Cash,Cancelled,TRK26707221,6,No Coupon,Email,1495.98,1495.98
2,ORD200469,45256,C13877,Chair,5,676.98,893 Main St,Cash,Cancelled,TRK17254691,5,No Coupon,Facebook,3384.90,3384.90
3,ORD200484,45441,C29092,Chair,4,15.01,767 Main St,Cash,Cancelled,TRK90455340,6,No Coupon,Facebook,60.04,60.04
4,ORD201151,44955,C94381,Chair,2,608.04,962 Main St,Cash,Cancelled,TRK69804304,4,No Coupon,Google,1216.08,1216.08



--- 2. FILTERING & SORTING (WHERE & ORDER BY) ---


,OrderID,Product,Quantity,TotalPrice
0,ORD200789,Tablet,5,3456.40
1,ORD201122,Monitor,5,3390.95
2,ORD200632,Laptop,5,3390.80
3,ORD200469,Chair,5,3384.90
4,ORD200328,Tablet,5,3370.20



--- 3. DATA AGGREGATIONS (SUM, AVG, COUNT) ---


,Total_Orders,Total_Revenue,Avg_Unit_Price
0,1200,1264761.96,356.41275



--- 4. GROUPING DATA (GROUP BY) ---


,Product,Units_Sold,Revenue
0,Chair,562,195620.11
1,Printer,542,195612.61
2,Laptop,535,192126.56
3,Tablet,497,186568.95
4,Monitor,480,175651.41
5,Desk,508,167459.93
6,Phone,411,151722.39


In [3]:
# Query 5: Deep dive into Product sales performance
product_query = """
SELECT
    Product,
    COUNT(OrderID) as Total_Orders,
    SUM(Quantity) as Total_Units_Sold,
    ROUND(SUM(TotalPrice), 2) as Total_Revenue,
    ROUND(AVG(TotalPrice), 2) as Average_Order_Value
FROM orders
GROUP BY Product
ORDER BY Total_Revenue DESC;
"""
print("📊 PRODUCT SALES & REVENUE PERFORMANCE:")
display(run_query(product_query))

📊 PRODUCT SALES & REVENUE PERFORMANCE:


,Product,Total_Orders,Total_Units_Sold,Total_Revenue,Average_Order_Value
0,Chair,178,562,195620.11,1098.99
1,Printer,181,542,195612.61,1080.73
2,Laptop,173,535,192126.56,1110.56
3,Tablet,179,497,186568.95,1042.28
4,Monitor,163,480,175651.41,1077.62
5,Desk,170,508,167459.93,985.06
6,Phone,156,411,151722.39,972.58


In [4]:
# Query 6: Revenue distribution by Payment Method
payment_query = """
SELECT
    PaymentMethod,
    COUNT(OrderID) as Order_Count,
    ROUND(SUM(TotalPrice), 2) as Total_Revenue
FROM orders
GROUP BY PaymentMethod
ORDER BY Total_Revenue DESC;
"""

# Query 7: Breakdown of Order Delivery Statuses
status_query = """
SELECT
    OrderStatus,
    COUNT(OrderID) as Number_of_Orders
FROM orders
GROUP BY OrderStatus
ORDER BY Number_of_Orders DESC;
"""

print("💳 REVENUE BY PAYMENT METHOD:")
display(run_query(payment_query))

print("\n📦 ORDER LOGISTICS & STATUS BREAKDOWN:")
display(run_query(status_query))

💳 REVENUE BY PAYMENT METHOD:


,PaymentMethod,Order_Count,Total_Revenue
0,Credit Card,234,263847.63
1,Online,258,262442.94
2,Cash,246,259786.29
3,Gift Card,230,246323.92
4,Debit Card,232,232361.18



📦 ORDER LOGISTICS & STATUS BREAKDOWN:


,OrderStatus,Number_of_Orders
0,Cancelled,250
1,Returned,247
2,Pending,237
3,Shipped,235
4,Delivered,231
